# ST-Reason · 06 Baselines without memory

Two baselines that answer the **same questions** as notebook 05, but without the CV memory:

| Run | Model | What it sees | What it measures |
| --- | --- | --- | --- |
| `main.BLIND.A`, `star.BLIND.A` | Qwen3-8B (4-bit), the same reasoner as 05 | the question and options only, no video | language bias: how much can be guessed from the wording alone |
| `main.VLM.F16`, `star.VLM.F16` | Qwen3-VL-4B (fp16), the same VLM that wrote the events in 03 | 16 frames sampled evenly from the video + the question and options | what a vision-language model gets by looking at the frames directly, with no memory or reasoning pipeline |

The plan named Qwen3-VL-8B for the frames baseline; the pilot measured it 12× slower in 4-bit on a T4, so the 4B model is used (stated in the paper). Using the same VLM as notebook 03 also makes the comparison cleaner: any difference comes from the memory and reasoning pipeline, not from a different vision model.

**Which questions.** The question shards listed in `SHARDS` (default 0 and 1, the shards answered in notebook 05). One person runs this notebook; it is short.

**Runtime:** GPU (T4). About 1 hour in total: the blind runs take a few minutes, the frames runs most of the time. The main cell stops cleanly after 110 minutes; run it again to continue. Answers are uploaded to `outputs/predictions/` as verified zips (`06_*.zip`), and answered questions are skipped after a restart.

In [3]:
# --- Parameters: the only cell to edit -----------------------------------------------------------
SHARDS = [0, 1]           # question shards to answer (the shards run in notebook 05)
N_SHARDS = 4
DO_BLIND = True           # blind LLM baseline (question + options only)
DO_VLM = True             # frames-only VLM baseline
N_FRAMES = 16             # frames per video for the VLM baseline
LIMIT_QUESTIONS = None    # e.g. 8 for a quick test (per run); None for the real run
SYNC_EVERY_MIN = 8        # upload answers to Drive at least this often (minutes)

In [4]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '06_baselines'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'
assert SHARDS and all(s in range(N_SHARDS) for s in SHARDS)
SHARD = min(SHARDS)                                         # used in file names only
MANIFEST_PATH = P['runs'] / f'manifest_06_shard{SHARD}.csv'
DRIVE_LOG_PATH = LOG_PATH                                   # log locally, copy to Drive with each upload
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
import pandas as pd, numpy as np

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


## 1. Questions

The same frozen sets and the same shard rule as notebook 05 (a fixed hash of the question ID), so the baselines answer exactly the questions that 05 answered.

In [5]:
def read_set(name):
    return pd.read_csv(P['subsets'] / f'{name}.csv', dtype={'video_id': str})
def shard_of(uid):
    return int(hashlib.sha1(str(uid).encode()).hexdigest(), 16) % N_SHARDS

def nextqa_questions(df):
    return [{'uid': r.uid, 'key': str(r.video_id), 'question': r.question.rstrip('?') + '?',
             'options': [str(getattr(r, f'a{i}')) for i in range(5)], 'gold': int(r.answer_idx),
             'type': r.type, 'zip': 'videos_nextqa_eval.zip', 'stem': str(r.video_id)} for r in df.itertuples(index=False)]
def star_questions(df):
    out = []
    for r in df.itertuples(index=False):
        n = int(getattr(r, 'n_options', 4) or 4)
        out.append({'uid': r.uid, 'key': f'{r.uid}__orig', 'question': str(r.question),
                    'options': [str(getattr(r, f'a{i}')).rstrip('.') for i in range(n)],
                    'gold': int(r.answer_idx), 'type': r.type, 'zip': 'videos_star_eval.zip', 'stem': f'{r.uid}__orig'})
    return out

QSETS = {'main': nextqa_questions(read_set('main_grid_eval')), 'star': star_questions(read_set('star_main'))}
for name in QSETS:
    QSETS[name] = [q for q in QSETS[name] if shard_of(q['uid']) in SHARDS]
    if LIMIT_QUESTIONS:
        QSETS[name] = QSETS[name][:LIMIT_QUESTIONS]
print({n: len(qs) for n, qs in QSETS.items()}, 'questions in shards', SHARDS)

def R(qset, fmt, strat):
    return {'id': f'{qset}.{fmt}.{strat}', 'qset': qset, 'format': fmt, 'strategy': strat}
RUNS = ([R(s, 'BLIND', 'A') for s in ['main', 'star']] if DO_BLIND else []) + \
       ([R(s, 'VLM', f'F{N_FRAMES}') for s in ['main', 'star']] if DO_VLM else [])
print(len(RUNS), 'runs:', ', '.join(r['id'] for r in RUNS))

{'main': 193, 'star': 190} questions in shards [0, 1]
4 runs: main.BLIND.A, star.BLIND.A, main.VLM.F16, star.VLM.F16


## 2. Saving answers (verified zips)

The same scheme as notebook 05: answers are written to the Colab disk, zipped every few minutes into `outputs/predictions/06_shard<k>_….zip`, and read back from Drive to verify. The answer files use the same fields as 05's, so notebook 08 scores both together.

In [6]:
LOCAL_PRED = P['local'] / 'pred_out_06'
LOCAL_PRED.mkdir(parents=True, exist_ok=True)
PRED_DRIVE = P['pred']
_PACKED = set()

class DriveSyncError(BaseException):
    pass

def restore_preds():
    n = 0
    for z in sorted(PRED_DRIVE.glob('06_*.zip'), reverse=True):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    out = LOCAL_PRED / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n += 1
        except (zipfile.BadZipFile, OSError):
            log(f'damaged zip skipped: {z.name}', 'WARN')
    log(f'restored {n} answer files from Drive')

def sync_preds():
    new = [str(p.relative_to(LOCAL_PRED)) for p in LOCAL_PRED.rglob('*.jsonl') if str(p.relative_to(LOCAL_PRED)) not in _PACKED]
    if not new:
        return 0
    name = f'06_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rel in new:
            zf.write(LOCAL_PRED / rel, rel)
    ok = False
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, PRED_DRIVE / name)
            with zipfile.ZipFile(PRED_DRIVE / name) as zf:
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            log(f'upload of {name} failed ({e.__class__.__name__}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. Answers are still on this Colab disk: do NOT restart; '
                             'run the summary cell (section 7) to retry the upload.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files, verified on Drive')
    return len(new)

def answered(run_id):
    out = set()
    for f in (LOCAL_PRED / run_id).glob('*.jsonl'):
        out |= {r['uid'] for r in read_jsonl(f)}
    return out

restore_preds()

[09:12:42] INFO  restored 0 answer files from Drive


## 3. Answer parsing and scoring

The blind run replies in JSON exactly as in 05 (the same parser, one retry on an invalid reply). The VLM is asked for the option letter only, which a 4B VLM follows more reliably than JSON; the first standalone letter A–E in its reply is taken. A reply with no usable letter counts as wrong and is listed in `valid_json` (the field name is kept from 05 so notebook 08 reads both alike).

In [7]:
LET = 'ABCDE'
def parse_json(text):
    """The first JSON object (dict) in a reply, or None. Lists and other JSON values are ignored."""
    if not isinstance(text, str):
        return None
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    s = text.find('{')
    while s != -1:
        depth = 0
        for j in range(s, len(text)):
            depth += (text[j] == '{') - (text[j] == '}')
            if depth == 0:
                try:
                    v = json.loads(text[s:j + 1])
                    if isinstance(v, dict):
                        return v
                except json.JSONDecodeError:
                    pass
                break
        s = text.find('{', s + 1)
    return None

def letter_of(js, q):
    if not isinstance(js, dict):
        return None
    a = js.get('answer', '')
    if isinstance(a, list):
        a = a[0] if a else ''
    a = str(a).strip().upper()[:1]
    return a if a and a in LET[:len(q['options'])] else None

def options_text(q):
    return '\n'.join(f'{LET[i]}. {o}' for i, o in enumerate(q['options']))

def letter_from_text(text, q):
    """The VLM's answer: the first standalone option letter in its reply."""
    if not isinstance(text, str):
        return None
    ok = LET[:len(q['options'])]
    m = re.search(r'\b([A-E])\b', text.strip().upper())
    return m.group(1) if m and m.group(1) in ok else None

def record(run, q, model_id, pred, raw, seconds, calls=1, extra=None):
    r = {'uid': q['uid'], 'run': run['id'], 'qset': run['qset'], 'format': run['format'], 'strategy': run['strategy'],
         'key': q['key'], 'type': q['type'], 'model': model_id, 'calls': calls, 'pred': pred, 'gold': LET[q['gold']],
         'correct': bool(pred == LET[q['gold']]), 'valid_json': pred is not None,
         'raw': None if pred is not None else (raw or '')[:300], 'seconds': round(seconds, 2)}
    r.update(extra or {})
    return r

def save_records(run, recs):
    out = LOCAL_PRED / run['id'] / f's{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S-%f}.jsonl'
    out.parent.mkdir(parents=True, exist_ok=True)
    for r in recs:
        append_jsonl(out, r)

## 4. Blind LLM baseline

Qwen3-8B (4-bit, thinking off), loaded exactly as in 05. The system prompt says plainly that the video is not available and asks for the most likely option.

In [8]:
if not MOCK:
    pip_install('transformers>=4.57.0', 'accelerate>=1.0', 'bitsandbytes>=0.45')
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()

def load_llm(spec):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    kw = {'device_map': {'': 0}, 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw['dtype'] = torch.float16
    tok = AutoTokenizer.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    try:
        model = AutoModelForCausalLM.from_pretrained(spec['id'], **kw)
    except (ValueError, KeyError):                       # Gemma 3 4B is multimodal
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    devs = {str(p.device) for p in model.parameters()}
    if any(not d.startswith('cuda') for d in devs):
        raise RuntimeError(f'model not fully on the GPU ({devs}): Disconnect and delete runtime, then Run all once.')
    return model.eval(), tok

_GEN_CAP = {}   # learned batch size per (model, task length): survives between calls, so it does not run out of memory again and again
def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
    """Batched chat generation (thinking off), token-budgeted, with OOM backoff. Returns (texts, prompt_lens, gen_lens)."""
    import torch, gc
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in convs]
    lens = [len(tok(t)['input_ids']) for t in texts]
    order = sorted(range(len(texts)), key=lambda k: lens[k])
    outs, glens = [None] * len(texts), [0] * len(texts)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    gen.update(dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False))
    key = (id(model), max_new_tokens)
    cap = min(batch_size, _GEN_CAP.get(key, batch_size))
    i, ok_streak = 0, 0
    while i < len(order):
        j = i + 1
        while j < len(order) and j - i < cap and (j - i + 1) * (lens[order[j]] + max_new_tokens) <= max_batch_tokens:
            j += 1
        idx = order[i:j]
        oom = False
        try:
            enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True).to(model.device)
            with torch.inference_mode():
                out = model.generate(**enc, **gen)
            new = out[:, enc['input_ids'].shape[1]:]
            for k, row, d in zip(idx, new, tok.batch_decode(new, skip_special_tokens=True)):
                outs[k], glens[k] = d, int((row != tok.pad_token_id).sum())
            del enc, out, new
            i = j
            if len(idx) > _GEN_CAP.get(key, batch_size):    # a bigger batch worked: remember it
                _GEN_CAP[key] = len(idx)
            ok_streak += 1
            if ok_streak >= 4 and cap < batch_size:        # try one bigger again after 4 good batches
                cap += 1; ok_streak = 0
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:                                           # free memory OUTSIDE the except block, or it stays held
            enc = out = new = None
            gc.collect(); torch.cuda.empty_cache()
            if len(idx) == 1:
                raise torch.cuda.OutOfMemoryError('one prompt alone does not fit on the GPU')
            cap = max(1, len(idx) // 2); ok_streak = 0
            _GEN_CAP[key] = cap
            log(f'GPU memory full at batch {len(idx)}: retrying with batch {cap}', 'WARN')
    return outs, lens, glens

if MOCK:
    class _T:
        pad_token_id = 0
        def __call__(self, t):
            return {'input_ids': t.split()}
        def apply_chat_template(self, m, **k):
            return ' '.join(x['content'] for x in m)
    def load_llm(spec):
        return None, _T()
    def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
        outs = []
        for m in convs:
            u = m[-1]['content']
            ids = re.findall(r'\b(E\d+|P\d+|C\d+)\b', m[-2]['content'] if len(m) > 2 else u) or ['E1']
            if 'supported' in u:
                outs.append('{"supported": %s}' % ('true' if len(u) % 3 else 'false'))
            elif 'sub_questions' in u:
                outs.append('{"type": "temporal", "target_object": "cup", "target_actor": "person", "anchor_event": "pick up", '
                            '"relation": "after", "sub_questions": ["What happens first?", "What happens next?"]}')
            else:
                letters = re.findall(r'^([A-E])\. ', u, flags=re.M) or ['A']
                outs.append('{"answer": "%s", "evidence": ["%s"], "reasoning_steps": ["checked"]}' % (letters[len(u) % len(letters)], ids[0]))
        return outs, [len(' '.join(x['content'] for x in m).split()) for m in convs], [20] * len(convs)

SYS_BLIND = ('You answer multiple-choice questions about a video. The video is NOT available to you: choose the most '
             'likely option from the question and the options alone. Reply with JSON only.')
FMT_BLIND = 'Reply as {"answer": "<letter>"}.'

def conv_blind(q):
    return [{'role': 'system', 'content': SYS_BLIND},
            {'role': 'user', 'content': f'QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n{FMT_BLIND}'}]

def answer_blind(qs):
    convs = [conv_blind(q) for q in qs]
    outs, _, _ = llm_generate(LLM, TOK, convs, 24, batch_size=32, max_batch_tokens=12000)
    res = [(letter_of(parse_json(o), q), o, 1) for o, q in zip(outs, qs)]
    bad = [i for i, (L, o, c) in enumerate(res) if L is None]
    if bad:
        retry = [convs[i] + [{'role': 'assistant', 'content': res[i][1] or ''},
                             {'role': 'user', 'content': 'That was not valid. ' + FMT_BLIND + ' JSON only.'}] for i in bad]
        outs2, _, _ = llm_generate(LLM, TOK, retry, 24, batch_size=32, max_batch_tokens=12000)
        for i, o in zip(bad, outs2):
            res[i] = (letter_of(parse_json(o), qs[i]), o, 2)
    return res

LLM = TOK = None
if DO_BLIND:
    BLIND_SPEC = CFG['models']['reasoner_main']
    if not MOCK:
        import torch
        free_memory()
    LLM, TOK = load_llm(BLIND_SPEC)
    print('blind reasoner:', BLIND_SPEC['id'], BLIND_SPEC.get('quant'))

[09:12:42] INFO  pip install transformers>=4.57.0 accelerate>=1.0 bitsandbytes>=0.45


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

blind reasoner: Qwen/Qwen3-8B nf4


## 5. Frames-only VLM baseline

Qwen3-VL-4B-Instruct in fp16, loaded as in notebook 03 (whole model on the GPU). For each video, 16 frames are taken at evenly spaced times (the centre of 16 equal parts of the video), resized to about 448×448 pixels, and given in time order with the question and options. Frames are taken once per video and reused for all its questions. The video is copied from its Drive zip to the Colab disk and deleted after use.

This cell only defines the functions; the model is loaded in section 6, after the blind runs have finished and the 8B model has been removed from the GPU (both do not fit together).

In [9]:
import cv2
from PIL import Image
VLM_SPEC = dict(CFG['models']['vlm_events'])          # Qwen3-VL-4B-Instruct, fp16 (the model used in 03)
VLM_MAX_PIXELS = 448 * 448
VLM_BATCH = {'n': 2}                                  # questions per call; halved if the GPU runs out of memory

def load_vlm(spec):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    kw = {'device_map': {'': 0}, 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main'), 'dtype': torch.float16}
    try:
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw).eval()
    except TypeError:                                 # older transformers: torch_dtype
        kw['torch_dtype'] = kw.pop('dtype')
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw).eval()
    devices = {str(p.device) for p in model.parameters()}
    if devices != {'cuda:0'}:
        raise RuntimeError(f'VLM not fully on the GPU ({devices}): Disconnect and delete runtime, then Run all once.')
    proc = AutoProcessor.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    proc.tokenizer.padding_side = 'left'
    return model, proc

def vlm_generate(model, proc, items, max_new_tokens=8):
    """items: [(images, prompt)] -> [reply text], one call for the whole batch."""
    import torch
    msgs = [[{'role': 'user', 'content': [*({'type': 'image', 'image': im} for im in ims),
                                          {'type': 'text', 'text': pr}]}] for ims, pr in items]
    inputs = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                      return_tensors='pt', padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

if MOCK:
    def load_vlm(spec):
        return None, None
    def vlm_generate(model, proc, items, max_new_tokens=8):
        return ['B' if len(pr) % 2 else 'The answer is C.' for ims, pr in items]

_ZIDX, _ZF = {}, {}
def zip_member(zname, stem):
    if zname not in _ZIDX:
        zp = P['data'] / zname
        if not zp.exists():
            _ZIDX[zname] = {}
        else:
            _ZF[zname] = zipfile.ZipFile(zp)
            _ZIDX[zname] = {Path(n).stem: n for n in _ZF[zname].namelist() if n.lower().endswith('.mp4')}
    return _ZIDX[zname].get(stem)

def to_pil(frame_bgr):
    im = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    w, h = im.size
    s = min(1.0, (VLM_MAX_PIXELS / (w * h)) ** 0.5)
    return im.resize((max(28, int(w * s)), max(28, int(h * s)))) if s < 1 else im

def sample_frames(zname, stem, n):
    """n frames at the centres of n equal parts of the video, in time order: (images, times in s)."""
    member = zip_member(zname, stem)
    if member is None:
        return None, None
    d = P['local_videos'] / 'vlm'; d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.mp4'
    with _ZF[zname].open(member) as s, open(path, 'wb') as f:
        shutil.copyfileobj(s, f)
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:                                    # frame count missing: count by reading once
        while cap.grab():
            total += 1
        cap.release(); cap = cv2.VideoCapture(str(path))
    want = sorted(set(((np.arange(n) + 0.5) * total / n).astype(int).clip(0, max(0, total - 1)).tolist()))
    ims, times, i, k = [], [], 0, 0
    while k < len(want) and cap.grab():
        if i == want[k]:
            ok, fr = cap.retrieve()
            if ok:
                ims.append(to_pil(fr)); times.append(round(i / fps, 1))
            k += 1
        i += 1
    cap.release()
    path.unlink(missing_ok=True)
    return ims, times

def vlm_prompt(q, times):
    span = f' (from {times[0]:.0f} s to {times[-1]:.0f} s)' if times else ''
    return (f'These are {len(times)} frames from one video, evenly spaced and in time order{span}.\n'
            f'QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n'
            'Answer with the letter of the correct option only.')

def answer_vlm(items):
    """Replies for [(images, prompt)], batched, with OOM backoff to one question per call."""
    outs, i = [None] * len(items), 0
    while i < len(items):
        chunk = items[i:i + VLM_BATCH['n']]
        oom = False
        try:
            outs[i:i + len(chunk)] = vlm_generate(VLM, PROC, chunk)
            i += len(chunk)
        except Exception as e:
            if MOCK or 'OutOfMemory' not in type(e).__name__:
                raise
            oom = True
        if oom:
            free_memory()
            if len(chunk) == 1:
                raise RuntimeError('one question with its frames does not fit on the GPU: lower N_FRAMES or VLM_MAX_PIXELS')
            VLM_BATCH['n'] = max(1, len(chunk) // 2)
            log(f'GPU memory full at batch {len(chunk)}: continuing with batch {VLM_BATCH["n"]}', 'WARN')
    return outs
VLM = PROC = None

## 6. Run

The blind runs go first (a few minutes), then the 8B model is removed from the GPU and the VLM is loaded for the frames runs. Answers are saved after every chunk of questions and uploaded every few minutes. The cell stops cleanly after 110 minutes; run it again to continue (answered questions are skipped).

In [10]:
RUN_LOG = Run(params={'step': 'baselines', 'shards': SHARDS, 'n_shards': N_SHARDS, 'runs': [r['id'] for r in RUNS],
                      'n_frames': N_FRAMES, 'limit': LIMIT_QUESTIONS})
budget_s = CFG['runtime']['chunk_minutes'] * 60
t_start, last_sync, stop = time.time(), time.time(), False
missing_video = {}

def maybe_sync():
    global last_sync
    if time.time() - last_sync > SYNC_EVERY_MIN * 60:
        sync_preds(); last_sync = time.time()

try:
    for run in RUNS:
        done = answered(run['id'])
        todo = [q for q in QSETS[run['qset']] if q['uid'] not in done]
        if not todo:
            continue
        log(f'{run["id"]}: {len(todo)} questions to do ({len(done)} already answered)')
        t_run = time.time()
        if run['format'] == 'BLIND':
            for c in range(0, len(todo), 64):
                if time.time() - t_start > budget_s:
                    stop = True; break
                chunk = todo[c:c + 64]
                t0 = time.time()
                res = answer_blind(chunk)
                per = (time.time() - t0) / len(chunk)
                save_records(run, [record(run, q, BLIND_SPEC['id'], L, o, per, calls=n) for q, (L, o, n) in zip(chunk, res)])
                maybe_sync()
        else:
            if VLM is None:
                if LLM is not None:                   # the 8B model and the VLM do not fit on the GPU together
                    free_memory('LLM', 'TOK'); LLM = TOK = None
                    free_memory()
                    # the Colab disk holds about 21 GB: remove the 8B download before fetching the VLM
                    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
                    shutil.rmtree(hub / ('models--' + BLIND_SPEC['id'].replace('/', '--')), ignore_errors=True)
                if not MOCK:
                    import torch
                    if torch.cuda.memory_allocated() / 1e9 > 1.0:
                        raise RuntimeError('GPU memory still in use: Disconnect and delete runtime, then Run all once '
                                           '(blind answers are already saved, so only the frames runs will run).')
                VLM, PROC = load_vlm(VLM_SPEC)
                log(f'frames baseline model: {VLM_SPEC["id"]}')
            by_video = {}
            for q in todo:
                by_video.setdefault((q['zip'], q['stem']), []).append(q)
            for (zname, stem), qs in by_video.items():
                if time.time() - t_start > budget_s:
                    stop = True; break
                t0 = time.time()
                ims, times = sample_frames(zname, stem, N_FRAMES)
                if not ims:
                    missing_video[run['id']] = missing_video.get(run['id'], 0) + len(qs)
                    continue
                replies = answer_vlm([(ims, vlm_prompt(q, times)) for q in qs])
                per = (time.time() - t0) / len(qs)
                save_records(run, [record(run, q, VLM_SPEC['id'], letter_from_text(rep, q), rep, per,
                                          extra={'n_frames': len(ims)}) for q, rep in zip(qs, replies)])
                maybe_sync()
        n_done = len(answered(run['id']) & {q['uid'] for q in QSETS[run['qset']]})
        recs = [r for f in (LOCAL_PRED / run['id']).glob('*.jsonl') for r in read_jsonl(f)]
        acc = np.mean([r['correct'] for r in recs]) if recs else float('nan')
        log(f'{run["id"]}: {n_done}/{len(QSETS[run["qset"]])} answered | '
            f'{(time.time() - t_run) / max(1, len(todo)):.1f} s per question | accuracy so far {acc:.3f}')
        if stop:
            log('time budget reached: run this cell again to continue', 'WARN')
            break
finally:
    sync_preds()
if missing_video:
    log(f'questions skipped because their video is not in its zip: {missing_video}', 'WARN')

[09:20:30] INFO  run started: 06_baselines-20261018-092030-cd4ac0
[09:20:30] INFO  main.BLIND.A: 193 questions to do (0 already answered)
[09:21:02] INFO  main.BLIND.A: 193/193 answered | 0.2 s per question | accuracy so far 0.420
[09:21:02] INFO  star.BLIND.A: 190 questions to do (0 already answered)
[09:21:31] INFO  star.BLIND.A: 190/190 answered | 0.2 s per question | accuracy so far 0.305
[09:21:31] INFO  main.VLM.F16: 193 questions to do (0 already answered)


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[09:24:34] INFO  frames baseline model: Qwen/Qwen3-VL-4B-Instruct
[09:28:32] INFO  uploaded 06_shard0_20261018-092832_00034.zip: 34 files, verified on Drive
[09:35:50] WARN  GPU memory full at batch 2: continuing with batch 1
[09:36:32] INFO  uploaded 06_shard0_20261018-093632_00062.zip: 62 files, verified on Drive
[09:41:40] INFO  main.VLM.F16: 193/193 answered | 6.3 s per question | accuracy so far 0.829
[09:41:40] INFO  star.VLM.F16: 190 questions to do (0 already answered)
[09:44:34] INFO  uploaded 06_shard0_20261018-094434_00079.zip: 79 files, verified on Drive
[09:52:35] INFO  uploaded 06_shard0_20261018-095235_00121.zip: 121 files, verified on Drive
[09:54:28] INFO  star.VLM.F16: 190/190 answered | 4.0 s per question | accuracy so far 0.758
[09:54:28] INFO  uploaded 06_shard0_20261018-095428_00026.zip: 26 files, verified on Drive


## 7. Progress and accuracy so far

Accuracies here are **not results yet**: notebook 08 scores everything together with confidence intervals. For context, chance is 0.20 on NExT-QA (5 options) and 0.25 on STAR (4 options).

In [11]:
try:
    sync_preds()
except DriveSyncError as e:
    log(str(e), 'ERROR')
rows = []
for run in RUNS:
    recs = [r for f in (LOCAL_PRED / run['id']).glob('*.jsonl') for r in read_jsonl(f)]
    rows.append({'run': run['id'], 'answered': len({r['uid'] for r in recs}), 'of': len(QSETS[run['qset']]),
                 'accuracy': round(np.mean([r['correct'] for r in recs]), 3) if recs else None,
                 'valid': round(np.mean([r['valid_json'] for r in recs]), 3) if recs else None,
                 's_per_q': round(np.mean([r['seconds'] for r in recs]), 2) if recs else None})
T = pd.DataFrame(rows)
print(T.to_string(index=False))
left = int((T['of'] - T['answered']).sum())
print(f'\n{left} questions left' + ('' if left else ': this notebook is finished'))
save_json({'shards': SHARDS, 'left': left, 'table': rows, 'updated': dt.datetime.now().isoformat(timespec='seconds')},
          P['pred'] / 'progress_06.json')
RUN_LOG.finish('done' if left == 0 else 'partial', notes=f'{left} questions left')

         run  answered  of  accuracy  valid  s_per_q
main.BLIND.A       193 193     0.420    1.0     0.16
star.BLIND.A       190 190     0.305    1.0     0.15
main.VLM.F16       193 193     0.829    1.0     5.31
star.VLM.F16       190 190     0.758    1.0     4.03

0 questions left: this notebook is finished
[09:54:28] INFO  run finished: 06_baselines-20261018-092030-cd4ac0 [done] 34.0 min, peak RAM 7.11 GB, peak VRAM 14.18 GB


## What to send back

The table from section 7. When it says `0 questions left`, this notebook is finished.